# IAT 461/882 — Assignment - Unsupervised Learning - Vancouver Business Licences Explorer

#### Submitted by:
Atif M. Mahmud  
atifm@sfu.ca

In [16]:
# Import the libraries
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as plt

## Part A - Business-level clustering

### Part A1 - Data acquisition and cleaning

#### Load and observe

In [17]:
gdf = gpd.read_file("data/business-licences.geojson")

print("The entire dataframe")
display(gdf)

print(f"\nThe shape of the dataframe is {gdf.shape}")
print(f"The columns of the dataframe are {gdf.columns.tolist()}")

print("\nDescriptive data")
display(gdf.describe())

# Create dataframe to list empty/nan/none vals
df_clean_explore = pd.DataFrame(index = gdf.columns, columns=["empty_string", "nan_vals", "none_vals"])

## NOTE: isna() and isnull() do the same thing, so I am using only one of them. I checked - no empty strings or "None" - or at least, they are handled by isna(). See commented code below
## To keep it simple, I will only use the isna()

# Null/Empty string/NaN/None: Fill in dataframe
# for column in gdf.columns:
#    df_clean_explore.at[column, "empty_string"] =  (gdf[column] == "").sum()
#    df_clean_explore.at[column, "nan_vals"] = gdf[column].isna().sum()
#    df_clean_explore.at[column, "none_vals"] = (gdf[column] == "None").sum()

print("\nGiven below: Column, NaN vals, Percentage of dataset")
rows = len(gdf)
for column in gdf.columns:
    nan_vals = gdf[column].isna().sum()
    percentage = (nan_vals/rows) * 100
    print(f"{column} | {nan_vals} | {percentage}%.")

The entire dataframe


,folderyear,licencersn,licencenumber,licencerevisionnumber,businessname,businesstradename,status,issueddate,expireddate,businesstype,...,province,country,postalcode,localarea,numberofemployees,feepaid,extractdate,geom,geo_point_2d,geometry
0,24,4518841,24-138560,10,Nobl Collective Ltd,NaN,Issued,2023-12-28 21:34:28+00:00,2024-12-31,Consulting and Management Services,...,BC,CA,NaN,Kitsilano,1.0,NaN,2026-07-01 02:32:17-07:00,None,NaN,None
1,24,4518843,24-138562,10,645064 BC Ltd,Trade Exchange Canada,Issued,2023-11-20 21:40:18+00:00,2024-12-31,Business Support Services,...,BC,CA,NaN,Kitsilano,1.0,NaN,2026-07-01 02:32:17-07:00,None,NaN,None
2,24,4518844,24-138563,10,(Louise Turgeon),Turgeon Business Consulting,Issued,2023-11-21 22:30:07+00:00,2024-12-31,Consulting and Management Services,...,BC,CA,NaN,Downtown,1.0,NaN,2026-07-01 02:32:17-07:00,None,NaN,None
3,24,4518848,24-138567,10,Baron Global Financial Canada Ltd,NaN,Issued,2023-12-11 18:12:12+00:00,2024-12-31,Consulting and Management Services,...,BC,CA,V6E 2E9,Downtown,5.0,NaN,2026-07-01 02:32:17-07:00,None,"{'lon': -123.118192793095, 'lat': 49.287734201...",POINT (-123.11819 49.28773)
4,24,4518856,24-138576,10,Jennifer D S Dezell (Jennifer Dezell),Dentons Canada LLP,Issued,2023-12-09 00:34:17+00:00,2024-12-31,Legal Services,...,BC,CA,V6C 3R8,Downtown,217.0,NaN,2026-07-01 02:32:17-07:00,None,"{'lon': -123.113252488858, 'lat': 49.286652613...",POINT (-123.11325 49.28665)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204254,26,4870638,26-148054,00,Shape Property Management Corp,NaN,Cancelled,NaT,NaT,Real Estate Services,...,BC,CA,V7X 1M6,Downtown,35.0,NaN,2026-07-25 00:09:01-07:00,None,"{'lon': -123.118959746657, 'lat': 49.286557071...",POINT (-123.11896 49.28656)
204255,26,4870639,26-148055,00,Bua Group Holdings Ltd,NaN,Issued,2026-01-19 17:29:13+00:00,2026-12-31,Real Estate Services,...,BC,CA,V6C 3A8,Downtown,0.0,324.0,2026-07-25 00:09:01-07:00,None,"{'lon': -123.11758014442, 'lat': 49.2860516234...",POINT (-123.11758 49.28605)
204256,26,4870640,26-148056,00,Shape Holdings Corp,NaN,Cancelled,NaT,NaT,Real Estate Services,...,BC,CA,V7X 1M6,Downtown,0.0,NaN,2026-07-25 00:09:01-07:00,None,"{'lon': -123.118959746657, 'lat': 49.286557071...",POINT (-123.11896 49.28656)
204257,26,4870655,26-148071,00,Coleman Enterprises Corp,NaN,Issued,2025-11-14 19:54:30+00:00,2026-12-31,Real Estate Services,...,BC,CA,V6C 1C8,Downtown,6.0,277.0,2026-07-25 00:09:01-07:00,None,"{'lon': -123.115954794154, 'lat': 49.286140186...",POINT (-123.11595 49.28614)



The shape of the dataframe is (204259, 26)
The columns of the dataframe are ['folderyear', 'licencersn', 'licencenumber', 'licencerevisionnumber', 'businessname', 'businesstradename', 'status', 'issueddate', 'expireddate', 'businesstype', 'businesssubtype', 'unit', 'unittype', 'house', 'street', 'city', 'province', 'country', 'postalcode', 'localarea', 'numberofemployees', 'feepaid', 'extractdate', 'geom', 'geo_point_2d', 'geometry']

Descriptive data


,expireddate,numberofemployees,feepaid
count,175515,204259.000000,128840.000000
mean,2025-12-30 18:59:53.908000,10.021531,516.801428
min,2024-01-11 00:00:00,0.000000,2.000000
25%,2024-12-31 00:00:00,0.000000,207.000000
50%,2025-12-31 00:00:00,1.000000,277.000000
75%,2026-12-31 00:00:00,4.000000,405.000000
max,2027-12-31 00:00:00,5876.000000,63722.000000
std,NaN,72.771582,1109.697928



Given below: Column, NaN vals, Percentage of dataset
folderyear | 0 | 0.0%.
licencersn | 0 | 0.0%.
licencenumber | 0 | 0.0%.
licencerevisionnumber | 0 | 0.0%.
businessname | 13482 | 6.6004435545067786%.
businesstradename | 126881 | 62.11770350388477%.
status | 0 | 0.0%.
issueddate | 28768 | 14.084079526483533%.
expireddate | 28744 | 14.072329738224509%.
businesstype | 0 | 0.0%.
businesssubtype | 182716 | 89.45309631399351%.
unit | 155310 | 76.0358172712096%.
unittype | 155575 | 76.16555451656964%.
house | 94481 | 46.25548935420226%.
street | 94464 | 46.24716658751879%.
city | 58 | 0.028395321625974863%.
province | 82 | 0.04014510988499895%.
country | 44985 | 22.023509368008266%.
postalcode | 95219 | 46.616795343167254%.
localarea | 2714 | 1.3287052222913067%.
numberofemployees | 0 | 0.0%.
feepaid | 75419 | 36.92322002947239%.
extractdate | 0 | 0.0%.
geom | 204259 | 100.0%.
geo_point_2d | 101627 | 49.75398880832668%.
geometry | 101627 | 49.75398880832668%.


#### Atif's thougts (for now)

- The entire column `geom` is empty. I will drop it because there is nothing useful we can do with it that won't be redundant.
- The `geo_point_2d` and `geometry` is a 1:1 mapping. Same number of missing values, same information, just in different format. Can drop either.
- There are 13482 mising `businessnames`, and 126881 missing `businesstradename`. Let's see if there are any where BOTH are missing. If not we can just use one or the other.

In [18]:
gdf.query("businessname.isna() and businesstradename.isna()")

,folderyear,licencersn,licencenumber,licencerevisionnumber,businessname,businesstradename,status,issueddate,expireddate,businesstype,...,province,country,postalcode,localarea,numberofemployees,feepaid,extractdate,geom,geo_point_2d,geometry
1156,24,4534547,24-223321,00,NaN,NaN,Pending,NaT,NaT,Short-term Rental Operator,...,BC,NaN,NaN,Sunset,0.0,NaN,2026-07-01 02:32:18-07:00,None,NaN,None
1197,24,4535874,24-224613,00,NaN,NaN,Issued,2024-05-07 20:50:09+00:00,2024-12-31,Short-term Rental Operator,...,BC,NaN,NaN,Hastings-Sunrise,0.0,737.0,2026-07-01 02:32:18-07:00,None,NaN,None
1207,24,4536420,24-225144,00,NaN,NaN,Issued,2024-05-08 17:48:13+00:00,2024-12-31,Short-term Rental Operator,...,BC,NaN,NaN,Mount Pleasant,0.0,737.0,2026-07-01 02:32:18-07:00,None,NaN,None
1214,24,4536671,24-225387,00,NaN,NaN,Pending,NaT,NaT,Short-term Rental Operator,...,BC,NaN,NaN,Mount Pleasant,0.0,NaN,2026-07-01 02:32:18-07:00,None,NaN,None
1217,24,4536734,24-225449,00,NaN,NaN,Cancelled,NaT,NaT,Short-term Rental Operator,...,BC,NaN,NaN,Downtown,0.0,70.0,2026-07-01 02:32:18-07:00,None,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193208,26,4976822,26-232882,00,NaN,NaN,Gone Out of Business,2026-05-11 16:33:39+00:00,2026-12-31,Short-term Rental Operator,...,BC,NaN,NaN,Dunbar-Southlands,0.0,816.0,2026-07-25 00:09:02-07:00,None,NaN,None
193212,26,4976932,26-232990,00,NaN,NaN,Issued,2026-05-09 00:32:07+00:00,2026-12-31,Short-term Rental Operator,...,BC,NaN,NaN,Grandview-Woodland,0.0,816.0,2026-07-25 00:09:02-07:00,None,NaN,None
193213,26,4977532,26-233586,00,NaN,NaN,Issued,2026-06-09 20:01:13+00:00,2026-12-31,Short-term Rental Operator,...,BC,NaN,NaN,Downtown,0.0,723.0,2026-07-25 00:09:02-07:00,None,NaN,None
193216,26,4977931,26-233984,00,NaN,NaN,Issued,2026-05-11 21:20:30+00:00,2026-12-31,Short-term Rental Operator,...,BC,NaN,NaN,Downtown,0.0,816.0,2026-07-25 00:09:02-07:00,None,NaN,None


#### Handling missing/"problematic" data

Since there are 13476, rows where BOTH `businessname` and `businesstradename` are missing, I will input `licensern` for them. One optional thing is to do it in levels: 
- First replace empty with an alias, something like `businessname` for `businesstradename` and vice-versa
- If even after, that empty or NaN values remain, use `licensern`

But here I am doing the second step directly